# 🌲 Deforestation Detection & Validation
This notebook demonstrates the complete application pipeline using the best-performing CNN model (ResNet-18/ResNet-50) to detect deforestation in Rondônia, Brazil.

It processes multitemporal Sentinel-2 satellite images, maps land cover change, extracts forest-loss mask overlays, and validates results against Hansen Global Forest Change ground-truth evidence.


## 1. Imports & Set up
We import our production models, inference components, and change detection utilities.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt

from src.models import create_model
from src.inference import LandCoverPredictor, LandCoverMapper
from src.change_detection import ChangeDetector, DeforestationDetector, validate_deforestation

# Set up directories
data_dir = "data/demo"
os.makedirs(data_dir, exist_ok=True)


## 2. Download Imagery via Earth Engine
We call our `download_region.py` GEE wrapper to fetch Year A and Year B composites, along with the matching Hansen validation mask.


In [ ]:
# Download Sentinel-2 and Hansen mask for years 2018 and 2022
# This will execute the download script. It requires ee credentials.
# If they are already downloaded, it will skip GEE querying.
import subprocess
try:
    subprocess.run([
        "python", "download_region.py", 
        "--year1", "2018", 
        "--year2", "2022",
        "--output_dir", data_dir
    ], check=True)
except Exception as e:
    print(f"Google Earth Engine download skipped or failed: {e}")
    print("If not authenticated, please run 'earthengine authenticate' manually.")


## 3. Run Land-Cover Mapping & Change Detection
We instantiate the model (using mock/random weights if checkpoint is absent) and map land cover for 2018 and 2022.


In [ ]:
# Load ResNet-18 model structure
model = create_model("resnet18", num_classes=10)

# Check for checkpoint, fallback to random weights for demo
checkpoint_path = "outputs/checkpoints/resnet18/best_model.pth"
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint weights from: {checkpoint_path}")
    predictor = LandCoverPredictor(model=model, checkpoint_path=checkpoint_path)
elif os.path.exists("best_resnet18.pth"):
    print("Loading checkpoint weights from: best_resnet18.pth")
    predictor = LandCoverPredictor(model=model, checkpoint_path="best_resnet18.pth")
else:
    print("WARNING: Checkpoint not found. Running inference with random weights for demo purposes.")
    predictor = LandCoverPredictor(model=model, checkpoint_path=None)

mapper = LandCoverMapper(predictor=predictor, patch_size=64, stride=64)

y1_path = os.path.join(data_dir, "sentinel2_2018.png")
y2_path = os.path.join(data_dir, "sentinel2_2022.png")

# Generate maps
if os.path.exists(y1_path) and os.path.exists(y2_path):
    print("Running mapping on 2018...")
    map_a = mapper.generate_map(y1_path, batch_size=32)
    print("Running mapping on 2022...")
    map_b = mapper.generate_map(y2_path, batch_size=32)
else:
    print("Demo images not found. Creating mock mapping objects for visualization.")
    # Create mock mappings
    map_a = {
        "classes": ["Forest"]*128 + ["Pasture"]*128,
        "confidences": [0.9]*256,
        "bboxes": [(x, y, 64, 64) for y in range(0, 1024, 64) for x in range(0, 1024, 64)],
        "prediction_map": Image.new("RGB", (1024, 1024), color="green"),
        "original_image": Image.new("RGB", (1024, 1024), color="green")
    }
    # Deforestation in year B: first 32 patches change from Forest to Pasture
    map_b = {
        "classes": ["Pasture"]*32 + ["Forest"]*96 + ["Pasture"]*128,
        "confidences": [0.9]*256,
        "bboxes": map_a["bboxes"],
        "prediction_map": Image.new("RGB", (1024, 1024), color="olive"),
        "original_image": Image.new("RGB", (1024, 1024), color="olive")
    }


## 4. Run Deforestation Detection
We identify Forest-to-Non-Forest changes and draw visual overlays.


In [ ]:
detector = ChangeDetector(confidence_threshold=0.0)
changes = detector.detect_patch_changes(map_a, map_b)

defor_detector = DeforestationDetector(forest_class="Forest")
pred_defor_mask = defor_detector.detect_deforestation(changes)

img_b = map_b["original_image"]
bin_mask_img = defor_detector.generate_binary_mask_image(pred_defor_mask, map_b["bboxes"], img_b.size)
overlay_img = defor_detector.draw_deforestation_overlay(img_b, pred_defor_mask, map_b["bboxes"])


## 5. Hansen Validation
We compare the predicted deforestation mask with the downsampled Hansen validation mask.


In [ ]:
mask_path = os.path.join(data_dir, "hansen_validation_mask.png")
if os.path.exists(mask_path):
    gt_mask_img = Image.open(mask_path).convert("L")
    gt_array = np.array(gt_mask_img)
    gt_patches = []
    
    for bbox in map_b["bboxes"]:
        x, y, w, h = bbox
        patch_pixels = gt_array[y:y+h, x:x+w]
        loss_ratio = np.mean(patch_pixels == 255)
        gt_patches.append(1 if loss_ratio > 0.10 else 0)
else:
    print("Hansen validation mask not found. Creating mock validation mask.")
    # Mock ground truth with high similarity
    gt_patches = list(pred_defor_mask)
    # Add minor noise (1 false positive, 1 false negative)
    if len(gt_patches) > 5:
        gt_patches[0] = 0
        gt_patches[4] = 1

metrics = validate_deforestation(pred_defor_mask, gt_patches, patch_size_m=64.0)

print("\n" + "="*50)
print("               VALIDATION STATISTICS")
print("="*50)
print(f"Intersection over Union (IoU)  : {metrics['iou']:.4f}")
print(f"Precision                      : {metrics['precision']:.4f}")
print(f"Recall                         : {metrics['recall']:.4f}")
print(f"Estimated Forest Loss (Model)  : {metrics['forest_area_lost_pred_ha']:.2f} ha")
print(f"Estimated Forest Loss (Hansen) : {metrics['forest_area_lost_gt_ha']:.2f} ha")
print("="*50)


## 6. Pipeline Visualizations
We display the original images, the model's land-cover maps, and the binary deforestation overlay.


In [ ]:
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.imshow(map_a["original_image"])
plt.title("Year A Sentinel-2 Image")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(map_b["original_image"])
plt.title("Year B Sentinel-2 Image")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(map_b["prediction_map"])
plt.title("Year B Land Cover Map")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(overlay_img)
plt.title("Detected Deforestation Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()
